In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re

In [2]:
BASE_URL = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"
year = 2024

res = requests.get(BASE_URL)
soup = BeautifulSoup(res.content, 'html.parser')

links = soup.find_all('a')
links = []

pattern = re.compile(r'.*Taxi Trip.*', re.IGNORECASE)
for link in soup.find_all('a', title=pattern):
    href = link.get('href')
    if href and str(year) in href:
        links.append(href)

In [3]:
yellow_links = [link for link in links if 'yellow' in link.lower()]
green_links = [link for link in links if 'green' in link.lower()]

In [4]:
yellow_data = pd.concat(
    [pd.read_parquet(link) for link in yellow_links],
    ignore_index=True
)
green_data = pd.concat(
    [pd.read_parquet(link) for link in green_links],
    ignore_index=True
)

In [5]:
# add a column to identify the taxi type
yellow_data['is_yellow'] = True
green_data['is_yellow'] = False

In [6]:
yellow_data.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee', 'is_yellow'],
      dtype='object')

In [7]:
green_data.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'is_yellow'],
      dtype='object')

In [8]:
yellow_data = yellow_data.rename(columns={
    'tpep_pickup_datetime': 'pickup_datetime',
    'tpep_dropoff_datetime': 'dropoff_datetime'
})
green_data = green_data.rename(columns={
    'lpep_pickup_datetime': 'pickup_datetime',
    'lpep_dropoff_datetime': 'dropoff_datetime'
})

In [9]:
yellow_data.columns

Index(['VendorID', 'pickup_datetime', 'dropoff_datetime', 'passenger_count',
       'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID',
       'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'Airport_fee', 'is_yellow'],
      dtype='object')

In [10]:
green_data.columns

Index(['VendorID', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag',
       'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count',
       'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount',
       'payment_type', 'trip_type', 'congestion_surcharge', 'is_yellow'],
      dtype='object')

In [11]:
combined_data = pd.concat([yellow_data, green_data], ignore_index=True)

In [12]:
combined_data.to_parquet("./data/taxi_data.parquet", index=False)